# Kontroller ve isteğe bağlı deneyler

Her oturumda tek aşama seçin: basic → full → isteğe bağlı wavelet → loss. Aşamalar aynı yeni workspace ve sürümü kullanır. Varsayılan eğitim kapalıdır. Bu, dondurulmuş protokolün yeni ortamda tekrar üretimidir; yeni yöntem protokolü değildir.

**Yayın:** advisor-notebooks-v2.1.0 · **Depo:** umutergul74/YeniBotDeney

**Girdi:** Drive `MyDrive/yeniBot/advisor_experiments/colab_v1` arşivi.
**Çıktı:** ayrı workspace; veriler `data/`, modeller `runs/`, raporlar `reports/`.
Drive hesabı: `icinbedava34@gmail.com`. Bağlarken hesabı kontrol edin.

**Başarı ölçütü:** işlem hatasız tamamlanır, `STATUS.md` ve inceleme paketi oluşur.
Bu teknik başarı, modelin iyi performans verdiği anlamına gelmez.

Hücreleri yukarıdan aşağı çalıştırın. İlk kurulum paket indirebilir.
Eğitim için T4 gerekir. Aynı workspace'e ikinci oturumdan yazmayın.
Kesintiden sonra aynı sürüm/ayarlarla baştan çalıştırın; uyumlu checkpointler kullanılır.
v2.0.x workspace'ine v2.1.0 ile devam etmeyin; yeni WORKSPACE_ID kullanın.


## 1 — Çalışma ayarları

Aynı deneye devam ederken WORKSPACE_ID ve sürümü değiştirmeyin.

| Aşama | Amaç | Önkoşul | Yeni eğitim sayısı |
|---|---|---|---:|
| basic | A: temel özellik/GRU kontrolü | Sabit ham arşiv | 114 |
| full | B/C/D: özellik ve mimari kontrolleri | Aynı workspace'te tamamlanmış A | 342 |
| wavelet | Nedensel filtre katkısı, isteğe bağlı | Tam A–D; desteklenen A kazananı | 114 |
| loss | Kayıp fonksiyonu katkısı, isteğe bağlı | A–D ve wavelet karşılaştırması | 456 |

Loss mevcut protokolde wavelet kararını gerektirir; wavelet atlanarak loss çalıştırılmaz.
08/09'un tarihsel sonucu wavelet kapalı + saf BCE'dir. Bu tüm yöntem ailesi için
üstünlük iddiası değildir. Aday seçimi validation BCE; Eylül 2026 testine dönülmez.
`ALLOW_TRAINING=False` hazırlığa izin verir; eğitim hücresi açıklama yazıp durur.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

MODE = 'research'  # recovery: yalnızca tablolar; research: yeni deney; archive: finali oku
PHASE = 'basic'  # @param ['basic', 'full', 'wavelet', 'loss']
WORKSPACE_ID = 'advisor_v21_research_001'
EXPERIMENTS = {
    'basic': ['A'],
    'full': ['B', 'C', 'D'],
    'wavelet': ['W_ON'],
    'loss': ['L_WEIGHTED_BCE', 'L_FOCAL', 'L_BCE_PEARSON', 'L_FOCAL_PEARSON'],
}[PHASE]
ALLOW_TRAINING = False
RELEASE_TAG = 'advisor-notebooks-v2.1.0'  # Sabit yayın; dalın güncel ucunu otomatik çekmez.
REPO_URL = 'https://github.com/umutergul74/YeniBotDeney.git'
REPO_DIR = Path('/content/advisor_deney_v21_repo')
RUNTIME_DIR = Path('/content/advisor_isolated_runtime')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot/advisor_experiments')
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
assert WORKSPACE_ID.replace('_','').replace('-','').isalnum()
assert MODE in ('recovery','research','archive')
assert PHASE != 'final' or MODE == 'archive', 'Görülmüş final testi yeniden çalıştırılmaz.'

print('Seçilen aşama:', PHASE, '| Deneyler:', EXPERIMENTS)
print('Planlanan fold × seed:', len(EXPERIMENTS) * 38 * 3, '| Eğitim izni:', ALLOW_TRAINING)


## 2 — Drive ve ayrı çıktı alanı

Hazırlanmış tablolar, ağırlıklar ve ilerleme kalıcı alana yazılır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
LEGACY_ROOT = DRIVE_BASE / 'colab_v1'
namespace = {'recovery':'recovery_v2','research':'research_v2','archive':'archive_audit_v2'}[MODE]
ROOT = DRIVE_BASE / namespace / WORKSPACE_ID
ROOT.mkdir(parents=True,exist_ok=True)
LOG_DIR = ROOT / 'sessions' / SESSION
LOG_DIR.mkdir(parents=True,exist_ok=True)
print('Eski arşiv (salt okunur):',LEGACY_ROOT)
print('Bu çalışmanın kalıcı alanı:',ROOT)
print('Oturum günlükleri:',LOG_DIR)


## 3 — Sabit Git yayını

Yerel değişiklik varsa otomatik silinmez; yanlış sürümle devam edilmez.


In [ ]:
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',RELEASE_TAG,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
def git(*args):
    return subprocess.check_output(['git','-C',str(REPO_DIR),*args],text=True).strip()
assert not git('status','--porcelain','--untracked-files=no'), 'Kodda yerel değişiklik var.'
assert git('rev-parse','HEAD') == git('rev-parse',RELEASE_TAG+'^{commit}'), 'Yanlış yayın; yeni runtime açın.'
print('Kod sürümü:',git('rev-parse','HEAD'))
assert git('remote', 'get-url', 'origin') == REPO_URL, 'Yanlış kaynak depo; yeni runtime açın.'


## 4 — Kilitli araştırma ortamı

Sistem Python'una pip install yapılmaz. Ayrı Python + venv kullanılır; bütün
bağımlılıklar sürüm ve SHA ile sabittir. Drive'daki indirme önbelleği tekrar kullanılır.
uv kilitleri ve paket onbellegi /content altinda tutulur. Drive kalici ciktilari
ve dogrulanmis kurucu dosyasini saklar. Yeni runtime paketleri yeniden indirebilir.
Kurulum başarısızsa sonraki hücreye geçmeyin. Eğitim için Tesla T4 seçin.


In [ ]:
def logged(command, name):
    with (LOG_DIR/name).open('a',encoding='utf-8',buffering=1) as log:
        env=dict(os.environ)
        for key in ('PYTHONPATH','PYTHONHOME','PYTHONSTARTUP','IPYTHONDIR','JUPYTER_CONFIG_DIR'):
            env.pop(key,None)
        env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
        env['MPLBACKEND']='Agg'  # Colab inline backend is host-only.
        with subprocess.Popen(command,cwd=REPO_DIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line,end=''); log.write(line)
                code=process.wait()
            except BaseException:
                process.terminate(); process.wait(); raise
            if code: raise subprocess.CalledProcessError(code,command)
RECEIPT=RUNTIME_DIR/'environment_receipt.json'
logged([sys.executable,'-I',str(REPO_DIR/'scripts/bootstrap_advisor.py'),
        '--repo',str(REPO_DIR),'--runtime',str(RUNTIME_DIR),
        '--cache',str(DRIVE_BASE/'reproducibility_cache_v2'),
        '--receipt',str(RECEIPT)],'environment_setup.log')
runtime=json.loads(RECEIPT.read_text())
RESEARCH_PYTHON=runtime['python']
print('İzole Python:',RESEARCH_PYTHON)
print('Paket kilidi:',runtime['lock_sha256'])

def execute_step(action):
    command=[RESEARCH_PYTHON,'-I',str(REPO_DIR/'scripts/advisor_workspace.py'),action,
             '--root',str(ROOT),'--legacy',str(LEGACY_ROOT),'--mode',MODE,
             '--phase',PHASE,'--session',SESSION,'--experiments',*EXPERIMENTS]
    if ALLOW_TRAINING: command.append('--allow-training')
    logged(command,action+'.log')


## 5 — Veriyi hazırla ve Drive'a kalıcı kaydet

Veri + manifest birlikte saklanır; `reports/prepared_frames.zip` gerçek
tabloları da içerir. Recovery modunda özgün tablo hash'i eşleşmezse durur.
Eski modeller veya tablolar değiştirilmez. Research modunda yeni kontrol
çalışmaları kendi namespace'inde hazırlanır. Kaynaklar eski sabit snapshot'tır.


In [ ]:
execute_step('prepare')


## 6 — Eğitim / kaldığı yerden devam

Recovery modunda eğitim yapılmaz. Research modunda ALLOW_TRAINING=True
seçilmelidir. Her epoch'un ağırlıkları, optimizer/RNG, en iyi model ve scaler
mevcut motor tarafından Drive'a yazılır. Uyumlu bitmiş foldlar atlanır.
Yeni ortamda kontrol sonuçları farklı çıkarsa eski kazanan zorlanmaz;
mevcut dar wavelet/loss protokolü durur ve yeni araştırma kararı gerekir.


In [ ]:
if ALLOW_TRAINING:
    execute_step('train')
else:
    print('Yalnızca hazırlık tamamlandı. Eğitim için ayarlarda ALLOW_TRAINING=True seçin.')


## 7 — İlerleme ve çıktıların yeri


In [ ]:
execute_step('status')
print('Durum:',ROOT/'STATUS.md')
print('Hazırlanmış tablolar:',ROOT/'data')
print('Checkpointler:',ROOT/'runs')


## 8 — İnceleme paketi ve tam dosya envanteri

Küçük inceleme ZIP'i modelleri içermez. Prepared ZIP'i tabloları içerir.
Ağırlıklar/raw veriler Drive klasöründedir; tam yedek için workspace
klasörünün tamamını indirin. `full_artifact_inventory.json` tüm dosyaların
hashlerini listeler; aktif oturum loglarini ayri belirtir. Runtime kapanırsa notebooku baştan çalıştırın.
Sert VM kapanmasından sonra `.writer_lease` kalmışsa başka yazıcı
olmadığını doğrulamadan kilidi kaldırmayın. Ortam/sürüm değişirse eski
çalışma sessizce devam etmez. Platformlar arasında bit eşitliği garanti değildir.


In [ ]:
execute_step('bundle')
print('İnceleme paketi:',ROOT/'reports/advisor_latest_review_bundle.zip')
print('Tam envanter:',ROOT/'reports/full_artifact_inventory.json')
